# 01: Time-Bin MPS Simulation - Complete Blueprint

## Overview

This notebook provides a comprehensive blueprint for simulating a quantum link using Time-Bin MPS (Matrix Product State) methods. The physical chain being simulated is:

```
Rb atom → 780nm photon → QFC → 1517nm → Fiber → Middle Station BSM
```

### Core Physical Concepts

1. **Time-Bin Discretization**: Continuous field is discretized into time bins $b_n = \frac{1}{\sqrt{\Delta t}}\int_{t_n}^{t_n+\Delta t} dt\, b(t)$
2. **Polarization Encoding**: Atom spin entangles with photon **polarization** (NOT just photon presence)
3. **MPS Structure**: Sites arranged as $S - A_1 - B_1 - A_2 - B_2 - \cdots$ ("conveyor belt")
4. **TEBD/TDVP Evolution**: Apply local gates/unitaries and use SVD truncation

---

## 1. Hilbert Space Definitions

### 1.1 Atom System Site (S)

Each atom ($j \in \{A,B\}$) is a 3-level system:
$$
\mathcal{H}_{\mathrm{atom},j} = \mathrm{span}\{|0\rangle_j, |1\rangle_j, |e\rangle_j\}
$$

- $|0\rangle, |1\rangle$: Two stable ground states (qubit)
- $|e\rangle$: Excited state for emission

Atomic jump operators:
$$
S_{j,+} = |0\rangle_j\langle e|, \quad S_{j,-} = |1\rangle_j\langle e|
$$

### 1.2 Time-Bin Field Sites

Each time-bin site ($A_n$ or $B_n$) has a **5D local basis**:
$$
\{ |\mathrm{vac}\rangle, |780,H\rangle, |780,V\rangle, |1517,H\rangle, |1517,V\rangle \}
$$

Discrete mode operators:
$$
b_{j,n}^p = \frac{1}{\sqrt{\Delta t}}\int_{t_n}^{t_n+\Delta t} dt\, b_j^p(t), \quad
c_{j,n}^p = \frac{1}{\sqrt{\Delta t}}\int_{t_n}^{t_n+\Delta t} dt\, c_j^p(t)
$$

Commutation: $[b_{j,n}^p, b_{j,m}^{p'\dagger}] = \delta_{nm}\delta_{pp'}$

### 1.3 Full MPS Site Ordering

$$
S - A_1 - B_1 - A_2 - B_2 - \cdots - A_N - B_N
$$

Where:
- $S$: Composite system site (two atoms + optional filter cavity modes)
- $A_n$: Arm A, bin n
- $B_n$: Arm B, bin n

**Why this ordering?** The beam splitter interference happens between adjacent $A_n$ and $B_n$, making it a natural two-site gate.

---

## 2. The $\alpha^{(\mathrm{emit})}$ Matrix: Polarization Mapping

### 2.1 Definition

The emission process maps atomic transitions ($\sigma^\pm$) to optical polarization (H/V):
$$
\alpha^{(\mathrm{emit})} = 
\begin{pmatrix}
\alpha_{H,+} & \alpha_{H,-} \\
\alpha_{V,+} & \alpha_{V,-}
\end{pmatrix}
$$

### 2.2 Construction from Geometry

1. **Coordinate system**: Quantization axis $\hat{z}$, collection direction $\mathbf{n}(\theta, \phi)$
2. **Dipole vectors**: $\mathbf{d}_\mu$ for $\mu \in \{+, -\}$
3. **Far-field transverse field**:
   $$
   \mathbf{E}_\mu(\mathbf{n}) \propto \mathbf{n} \times (\mathbf{n} \times \mathbf{d}_\mu) = \mathbf{d}_\mu - \mathbf{n}(\mathbf{n} \cdot \mathbf{d}_\mu)
   $$
4. **Project onto polarization basis** $\{\mathbf{e}_1, \mathbf{e}_2\}$:
   $$
   |u_\mu\rangle = \begin{pmatrix} \mathbf{e}_1 \cdot \mathbf{E}_\mu \\ \mathbf{e}_2 \cdot \mathbf{E}_\mu \end{pmatrix}
   $$
5. **Apply basis rotation** if needed: $\begin{pmatrix}\alpha_{H,\mu} & \alpha_{V,\mu}\end{pmatrix} = R \frac{|u_\mu\rangle}{|u_\mu|}$
6. **Include channel imbalance**: $\alpha_{p,\mu} \leftarrow \sqrt{\beta_\mu} e^{i\varphi_\mu} \alpha_{p,\mu}$

### 2.3 Entanglement Quality Check

For high-quality polarization entanglement, the two columns of $\alpha^{(\mathrm{emit})}$ should be nearly orthogonal:
$$
\langle u_+ | u_- \rangle = \frac{\sin^2\theta}{1 + \cos^2\theta}
$$

- $\theta = 0$: Orthogonal (good entanglement)
- $\theta = \pi/2$: Completely indistinguishable (no entanglement)

---

## 3. Gate Library

### 3.1 Emission Gate $U^{(\mathrm{emit})}_{j,n}$

**Type**: Two-site unitary on $(S, j_n)$

Coupling operator:
$$
L_{j,p}(t) = \sqrt{\gamma_j(t)} \left( \alpha_{p,+} S_{j,+} + \alpha_{p,-} S_{j,-} \right)
$$

Gate:
$$
U^{(\mathrm{emit})}_{j,n} = \exp\left[ \sqrt{\Delta t} \sum_{p \in \{H,V\}} \left( L_{j,p}(t_n) b_{j,n}^{p\dagger} - \mathrm{h.c.} \right) \right]
$$

### 3.2 QFC Gate $U^{(\mathrm{QFC})}_{j,n}$

**Type**: One-site unitary on $j_n$

$$
U^{(\mathrm{QFC})}_{j,n} = \exp\left[ -i \sum_{p \in \{H,V\}} \theta_p \left( b_{j,n}^p c_{j,n}^{p\dagger} + \mathrm{h.c.} \right) \right]
$$

- $\sin^2\theta_p$: Conversion probability $780 \to 1517$
- Can be polarization-dependent ($\theta_H \neq \theta_V$)

### 3.3 Filter Cavity Gate $U^{(\mathrm{filt})}_{j,n}$

**Type**: Two-site unitary on $(S, j_n)$ (cavity mode $a_{j,p}$ in $S$)

$$
U^{(\mathrm{filt})}_{j,n} \approx \exp\left[ \sqrt{\Delta t} \sum_p \left( \sqrt{\kappa_f} a_{j,p} c_{j,n}^{p\dagger} - \mathrm{h.c.} \right) \right] e^{-i H_f \Delta t / \hbar}
$$

This creates memory effects and entanglement between adjacent bins.

### 3.4 Loss/Damping (Kraus Channel)

**Type**: One-site Kraus on $j_n$

For a single mode with transmission $\eta$:
$$
K_0 = |0\rangle\langle 0| + \sqrt{\eta}|1\rangle\langle 1|, \quad
K_1 = \sqrt{1-\eta}\, |0\rangle\langle 1|
$$

For PDL: use different $\eta_H, \eta_V$.

### 3.5 Jones Polarization Rotation $U^{(\mathrm{pol})}_j$

**Type**: One-site unitary

$$
\begin{pmatrix} c_{j,n}^H \\ c_{j,n}^V \end{pmatrix} \longrightarrow U_j \begin{pmatrix} c_{j,n}^H \\ c_{j,n}^V \end{pmatrix}, \quad U_j \in SU(2)
$$

In compressed basis: $U^{(\mathrm{pol})} = |\mathrm{vac}\rangle\langle\mathrm{vac}| + \sum_{p,q} (U_j)_{pq} |p\rangle\langle q|$

### 3.6 PMD (Polarization Mode Dispersion)

**Type**: Cross-bin operation using SWAP chain

PMD causes differential group delay between two PSPs:
$$
U_j(\omega) = R_{\mathrm{out}} \begin{pmatrix} e^{-i\omega\tau_1} & 0 \\ 0 & e^{-i\omega\tau_2} \end{pmatrix} R_{\mathrm{in}}
$$

With $m = \mathrm{round}(\Delta\tau/\Delta t)$:
1. Transform each bin: $R_{\mathrm{in}}$ (H/V $\to$ PSP) - one-site
2. Shift PSP$_2$ component by $m$ bins using nearest-neighbor SWAPs - two-site chain
3. Transform back: $R_{\mathrm{out}}^{-1}$ - one-site

### 3.7 Beam Splitter Gate $U^{(\mathrm{BS})}_n$

**Type**: Two-site unitary on $(A_n, B_n)$

$$
\begin{pmatrix} d_{1,n}^p \\ d_{2,n}^p \end{pmatrix} = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix} \begin{pmatrix} c_{A,n}^p \\ c_{B,n}^p \end{pmatrix}
$$

**CRITICAL**: Your local basis must allow two photons for correct BSM results (HOM bunching).

### 3.8 Detection Kraus

**Type**: Two-site measurement on $(A_n, B_n)$

For each detector mode $q$ with efficiency $\eta_d$ and dark count $p_{\mathrm{dc}}$:
$$
E_0^{(q)} = \sum_{m \geq 0} (1-p_{\mathrm{dc}})(1-\eta_d)^m |m\rangle\langle m|, \quad
E_1^{(q)} = I - E_0^{(q)}
$$

---

## 4. Execution Flow (Per Bin n)

### Initial Chain State:
$$
\cdots - B_{n-1} - S - A_n - B_n - A_{n+1} - \cdots
$$

### Step-by-Step:

| Step | Operation | Type | Sites |
|------|-----------|------|-------|
| 0 | System evolution + decoherence | 1-site unitary + Kraus | S |
| 1 | A-arm emission | 2-site unitary | (S, A_n) |
| 2 | A-arm QFC | 1-site unitary | A_n |
| 3 | A-arm filter | 2-site unitary | (S, A_n) |
| 4 | A-arm loss/noise | 1-site Kraus | A_n |
| 5 | A-arm Jones rotation | 1-site unitary | A_n |
| 6 | SWAP S $\leftrightarrow$ A_n | 2-site SWAP | (S, A_n) |
| 7 | B-arm emission | 2-site unitary | (S, B_n) |
| 8 | B-arm QFC | 1-site unitary | B_n |
| 9 | B-arm filter | 2-site unitary | (S, B_n) |
| 10 | B-arm loss/noise | 1-site Kraus | B_n |
| 11 | B-arm Jones rotation | 1-site unitary | B_n |
| 12 | SWAP S $\leftrightarrow$ B_n | 2-site SWAP | (S, B_n) |
| 13 | PMD sweep (optional) | 1-site + 2-site chain | A_k/B_k |
| 14 | Beam splitter interference | 2-site unitary | (A_n, B_n) |
| 15 | Detection + recording | 2-site measurement | (A_n, B_n) |
| 16 | Discard measured bins | - | - |

### After Step 12, chain state:
$$
\cdots - A_n - B_n - S - A_{n+1} - B_{n+1} - \cdots
$$

Now $(A_n, B_n)$ are adjacent and ready for BS interference!

---

## 5. Output Metrics

After $N$ bins, compute:

1. **Success probability**:
   $$
   p_{\mathrm{succ}} = \Pr(\mathbf{r}_{1:N} \in \mathcal{S})
   $$
   where $\mathbf{r}_n$ is the detector click pattern for bin $n$.

2. **Conditional atomic state**:
   $$
   \rho_{AB|\mathrm{succ}} = \frac{\operatorname{Tr}_{\mathrm{field}}[M_{\mathrm{succ}} \rho_{\mathrm{tot}} M_{\mathrm{succ}}^\dagger]}{p_{\mathrm{succ}}}
   $$

3. **Fidelity**:
   $$
   F = \langle \Phi_{\mathrm{target}} | \rho_{AB|\mathrm{succ}} | \Phi_{\mathrm{target}} \rangle
   $$

---

## 6. Implementation Checklist

### Source Emission
- [ ] $S_{j,\pm}$ jump operators
- [ ] $\gamma(t)$ pulse shaping
- [ ] $\alpha^{(\mathrm{emit})}$ matrix from geometry
- [ ] Channel imbalance $\beta_\pm$ and phase $\phi_j$

### QFC
- [ ] Conversion angle $\theta_p$ (polarization-dependent)
- [ ] Insertion loss/PDL $\eta_{\mathrm{ins}}^{(p)}$
- [ ] Noise equivalent background

### Filter
- [ ] Cavity mode $a_{j,p}$ with truncation
- [ ] $\kappa_f, \Delta_f$ parameters
- [ ] Cavity loss channel

### Fiber Channel
- [ ] Amplitude damping $\eta(L) = e^{-\alpha L}$
- [ ] Jones matrices $U_A, U_B$
- [ ] PMD: PSP basis + cross-bin shift

### Middle Station
- [ ] 50/50 BS gate
- [ ] PBS/routing to detectors
- [ ] Detection efficiency $\eta_d$, dark count $p_{\mathrm{dc}}$
- [ ] Success pattern set $\mathcal{S}$
- [ ] Coincidence window definition

### Numerics
- [ ] MPS state management (bond dimension, truncation)
- [ ] TEBD two-site update with SVD
- [ ] Trajectory method for Kraus operators
- [ ] Bin discard mechanism

---

## 7. Common Pitfalls

1. **BSM truncation closure**: With $|1\rangle_A |1\rangle_B$ input, after BS you get $|2\rangle$ components. If your local space doesn't include $|2\rangle$, HOM/BSM probabilities will be WRONG.

2. **Polarization chain consistency**: Test with $U_A = U_B$ and orthogonal $\alpha^{(\mathrm{emit})}$ columns to verify ideal Bell state recovery.

3. **High $p_{\mathrm{succ}}$ but low F**: Usually means:
   - $\alpha^{(\mathrm{emit})}$ columns non-orthogonal (source issue)
   - PMD reducing indistinguishability (HOM visibility)
   - Noise/dark counts causing false coincidences

---

Next: Implement individual gate types and test them on simple states.

---

## 8. TEBD Emission Test (from test_tebd_emission.py)

Run the following cell to execute the standalone emission test.


In [ ]:
"""
Simple TEBD Test: Atom Emission (Using L0 Layer MPSState)

Physics:
---------
Atom (3-level):
  |e>: 5P_{3/2}, F'=0, m_F=0  (excited)
  |0>: 5S_{1/2}, F=1, m_F=+1 (ground)
  |1>: 5S_{1/2}, F=1, m_F=-1 (ground)

Selection rules:
  |e> → |0>: Δm = +1 → σ+ photon (right circular)
  |e> → |1>: Δm = -1 → σ- photon (left circular)

780nm subspace (3D): |vac>, |H>, |V>
1517nm subspace (6D): |vac>, |H>, |V>, |2H>, |2V>, |HV>

Purpose:
--------
This test demonstrates the use of the L0 layer MPSState class for
applying TEBD-style gates to a quantum system.
"""

import sys
from pathlib import Path
import numpy as np

# Add parent directory to path for imports
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

# Import L0 layer (core) MPS infrastructure
from atom_sim.core.mps import MPSState

# ========================================================================
# Part 1: System Setup
# ========================================================================

print("=" * 70)
print("Part 1: System Setup")
print("=" * 70)

print("\nAtom basis (3D): |0> [idx 0], |1> [idx 1], |e> [idx 2]")
print("780nm photon basis (3D): |vac> [idx 0], |H> [idx 1], |V> [idx 2]")
print("1517nm photon basis (6D): not used in this test")

# ========================================================================
# Part 2: Build Emission Gate (U_emit)
# ========================================================================

print("\n" + "=" * 70)
print("Part 2: Build Emission Gate (U_emit)")
print("=" * 70)

# Gate parameters
dt = 0.1
g = 1.0  # coupling strength
theta = np.sqrt(dt) * g

# Dimensions
d_atom = 3
d_photon = 3
d_combined = d_atom * d_photon

print(f"\nGate construction: |e, vac> -> cos(theta)|e, vac> + sin(theta)/sqrt(2)(|0, H> + |1, V>)")
print(f"  theta = sqrt(dt) * g = {theta:.4f}")

# Build U_emit matrix
U_emit = np.eye(d_combined, dtype=complex)

# KEY: Index mapping for combined space (row-major ordering)
# Joint state |atom, photon> is flattened as: idx = atom_idx * d_photon + photon_idx
# Example: |e, vac> where |e>=idx 2, |vac>=idx 0 -> combined_idx = 2*3 + 0 = 6
e_vac_idx = 2 * d_photon + 0  # |e, vac>
target_H_idx = 0 * d_photon + 1  # |0, H>
target_V_idx = 1 * d_photon + 2  # |1, V>

print(f"\nCombined space indices (9D):")
print(f"|e, vac> → idx {e_vac_idx}")
print(f"|0, H>   → idx {target_H_idx}")
print(f"|1, V>   → idx {target_V_idx}")

# Set matrix elements: U[target, source] = amplitude
# U_emit[i, j] gives the amplitude for transition from state j to state i
U_emit[e_vac_idx, e_vac_idx] = np.cos(theta)           # |e, vac> -> |e, vac>
U_emit[target_H_idx, e_vac_idx] = np.sin(theta) / np.sqrt(2)  # |e, vac> -> |0, H>
U_emit[target_V_idx, e_vac_idx] = np.sin(theta) / np.sqrt(2)  # |e, vac> -> |1, V>

print("\nEmission gate U^(emit) (9×9):")
print("Row/Col: ", end="")
for i in range(d_combined):
    print(f"{i:4d}", end="")
print()
for i in range(d_combined):
    print(f"{i:4d}", end="")
    for j in range(d_combined):
        val = U_emit[i, j]
        if abs(val) > 1e-10:
            print(f"{val.real:4.1f}", end=" ")
        else:
            print("   .", end=" ")
    print()

# ========================================================================
# Part 3: Create Initial MPS State
# ========================================================================

print("\n" + "=" * 70)
print("Part 3: Create Initial MPS State")
print("=" * 70)

# Create MPS with 2 sites: atom (3D) and photon (3D)
# init_state[i] = index of local basis state for site i
# init_state=[2, 0] means: site 0 in |e> (idx 2), site 1 in |vac> (idx 0)
print("\nCreating MPS with local_dims = [3, 3]...")
print("Initial state: atom in |e> (index 2), photon in |vac> (index 0)")

mps = MPSState(local_dims=[d_atom, d_photon], init_state=[2, 0], max_bond=10)

print(f"MPS created:")
print(f"  L = {mps.L} sites")
print(f"  d = {mps.d}")
print(f"  chi = {mps.get_bond_dimensions()}")
print(f"  norm = {mps.norm():.6f}")

# Verify by extracting the atom state
rho_atom_init = mps.get_atom_state(system_site=0)
print("\nAtomic density matrix (should show |e><e| at index 2):")
print(rho_atom_init)

# ========================================================================
# Part 4: Apply Emission Gate using TEBD
# ========================================================================

print("\n" + "=" * 70)
print("Part 4: Apply Emission Gate via TEBD")
print("=" * 70)

print("\nApplying emission gate U^(emit) to sites (0, 1)...")
print("\nNote: TeNPy internally handles the tensor contraction, SVD, and canonicalization.")
print("We just call apply_two_site_gate() with the gate matrix.")
print()

# Apply the emission gate using TEBD
# The gate acts on sites 0 (atom) and 1 (photon)
mps.apply_two_site_gate(site_left=0, gate=U_emit, truncate=True)

print("Emission gate applied!")
print(f"Bond dimensions after gate: {mps.get_bond_dimensions()}")
print(f"State norm: {mps.norm():.6f}")

# Analyze the result
print("\n" + "-" * 70)
print("Analysis (extracted from MPS):")
print("-" * 70)

# Extract reduced density matrices from MPS
rho_atom = mps.get_atom_state(system_site=0)
print("\nAtomic reduced density matrix ρ_atom:")
print(rho_atom)
print(f"Trace: {np.trace(rho_atom).real:.6f}")

# Extract photon state (trace over atom)
# For photon site, we need to compute the reduced density matrix
# In a simple 2-site case, this is straightforward
rho_photon = mps.get_reduced_density(sites=[1])
print("\nPhoton reduced density matrix ρ_photon (780nm):")
print(rho_photon)
print(f"Trace: {np.trace(rho_photon).real:.6f}")

# Compute probabilities from diagonal elements
# Atom basis: |0>, |1>, |e> (indices 0, 1, 2)
# Photon basis: |vac>, |H>, |V> (indices 0, 1, 2)

# Probability that atom remains in |e>
p_remain = rho_atom[2, 2].real

# Probability of atom in |0> (H emitted)
p_H = rho_atom[0, 0].real

# Probability of atom in |1> (V emitted)
p_V = rho_atom[1, 1].real

p_emit = p_H + p_V

print(f"\nProbabilities from atom density matrix:")
print(f"  P(atom in |e>) = {p_remain:.6f}")
print(f"  P(atom in |0>, H emitted) = {p_H:.6f}")
print(f"  P(atom in |1>, V emitted) = {p_V:.6f}")
print(f"  P(emission occurred) = {p_emit:.6f}")

# Photon state probabilities
p_photon_vac = rho_photon[0, 0].real
p_photon_H = rho_photon[1, 1].real
p_photon_V = rho_photon[2, 2].real

print(f"\nProbabilities from photon density matrix:")
print(f"  P(|vac>) = {p_photon_vac:.6f}")
print(f"  P(|H>)   = {p_photon_H:.6f}")
print(f"  P(|V>)   = {p_photon_V:.6f}")

# Photon number expectation
n_op = np.array([[0, 0, 0],
                  [0, 1, 0],
                  [0, 0, 1]], dtype=complex)
n_exp = np.real(np.trace(rho_photon @ n_op))
print(f"\nExpected photon number: {n_exp:.6f}")

# Verify consistency
print(f"\nConsistency check:")
print(f"  Emission probability (atom)   = {p_emit:.6f}")
print(f"  Emission probability (photon) = {p_photon_H + p_photon_V:.6f}")
print(f"  Difference = {abs(p_emit - (p_photon_H + p_photon_V)):.2e}")

# ========================================================================
# Part 5: Entanglement Analysis (Bond Dimension)
# ========================================================================

print("\n" + "=" * 70)
print("Part 5: Entanglement Analysis")
print("=" * 70)

print("\nThe TEBD algorithm performs SVD internally during gate application.")
print("After the emission gate, the MPS bond dimension reflects entanglement")
print("between the atom and photon.")

print(f"\nBond dimensions: {mps.get_bond_dimensions()}")
chi_list = mps.get_bond_dimensions()
if len(chi_list) >= 3:
    print(f"  chi_0 (left boundary)  = {chi_list[0]}")
    print(f"  chi_1 (atom-photon)   = {chi_list[1]}")
    print(f"  chi_2 (right boundary) = {chi_list[2]}")
    bond_dim = chi_list[1]
elif len(chi_list) == 1:
    # For 2-site MPS, only one internal bond
    print(f"  chi (atom-photon bond) = {chi_list[0]}")
    bond_dim = chi_list[0]
else:
    bond_dim = chi_list[0] if chi_list else 0

print(f"\nSchmidt rank (bond dimension) = {bond_dim}")
print(f"This equals the number of terms in the Schmidt decomposition.")
print(f"After emission: |psi> = cos|e,vac> + sin/sqrt(2)|0,H> + sin/sqrt(2)|1,V>")
print(f"These 3 terms are orthogonal across the atom-photon bipartition,")
print(f"so the Schmidt rank = 3.")

print("\n" + "=" * 70)
print("Summary: L0 Layer Functions Used")
print("=" * 70)
print("\nThis test demonstrated the following L0 layer (atom_sim.core) functions:")
print("  1. MPSState.__init__()         - Create MPS with specified local dimensions")
print("  2. MPSState.apply_two_site_gate() - Apply two-site unitary (TEBD)")
print("  3. MPSState.get_atom_state()      - Extract reduced density matrix")
print("  4. MPSState.get_reduced_density() - Get any site's reduced state")
print("  5. MPSState.norm()                - Compute state norm")
print("  6. MPSState.get_bond_dimensions() - Get entanglement info")

print("\n" + "=" * 70)
print("Test Complete!")
print("=" * 70)

